In [33]:
import pandas as pd
messages  = pd.read_csv("Data\spam.csv",sep =',', names =["label","message"], skiprows = 1)

In [34]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [35]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sharm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [36]:
# Data Cleaning and Preprocessing
corpus = []
for i in range(0,len(messages)):
    review = re.sub('[^a-zA-Z]',' ', messages['message'][i]) # removing special characters
    review = review.lower() # lower casing the corpus
    review = review.split() 
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')] # Stopword removal and stemming
    review = ' '.join(review)
    corpus.append(review)

In [37]:
# Output feature
y = pd.get_dummies(messages['label'])
y =y.iloc[:,0].values

In [38]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus,y,test_size=0.20)

In [39]:
# Creating Bag of Words Model
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=100) # taking vocab with top frequency 2500 # for binary BOW enable binary = TRUE
X_train = cv.fit_transform(X_train).toarray()
X_test = cv.transform(X_test).toarray()

In [40]:
import numpy as np
np.set_printoptions(edgeitems=30,linewidth=100000,formatter=dict(flot = lambda x:"%.3g" % X_train)) # to display tohe vector neatly
X_train

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..., 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0

In [41]:
from sklearn.naive_bayes import MultinomialNB
spam_bow_detect_model = MultinomialNB().fit(X_train,y_train)

In [42]:
y_pred = spam_bow_detect_model.predict(X_test)

In [43]:
# perfromance metrics
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9506726457399103
              precision    recall  f1-score   support

       False       0.84      0.80      0.82       158
        True       0.97      0.97      0.97       957

    accuracy                           0.95      1115
   macro avg       0.90      0.89      0.90      1115
weighted avg       0.95      0.95      0.95      1115



## TF-IDF

In [44]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()

In [45]:
# # Text Preprocessing
# corpus = []
# for i in range(0,len(messages)):
#     review = re.sub('[^a-zA-Z]',' ', messages['message'][i]) # removing special characters
#     review = review.lower() # lower casing the corpus
#     review = review.split() 
#     review = [wnl.lemmatize(word) for word in review if not word in stopwords.words('english')] # Stopword removal and stemming
#     review = ' '.join(review)
#     corpus.append(review)

In [46]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(corpus,y,test_size=0.20)

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=2500,ngram_range=(1,2))
X_train = tfidf.fit_transform(X_train).toarray()
X_test = tfidf.transform(X_test).toarray()

In [48]:
from sklearn.naive_bayes import MultinomialNB
spam_tfidf_detect_model = MultinomialNB().fit(X_train,y_train)

In [49]:
y_pred = spam_tfidf_detect_model.predict(X_test)

In [50]:
# perfromance metrics
from sklearn.metrics import accuracy_score,classification_report
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.9757847533632287
              precision    recall  f1-score   support

       False       1.00      0.84      0.91       165
        True       0.97      1.00      0.99       950

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.97      1115



In [51]:
# 